# 11 · Pruning (Additional Experiment)

Applies unstructured L1 magnitude pruning to the best KD-trained student
at multiple sparsity levels, followed by short fine-tuning.

## Important limitation

**Unstructured pruning does not reduce inference latency on STM32.**
The X-CUBE-AI runtime executes dense matrix operations — zero-valued weights
do not skip computation. Pruning reduces the number of non-zero parameters
and on-disk model size, but deployed inference time and RAM usage are unchanged.

This experiment demonstrates accuracy degradation as a function of sparsity,
and confirms that **quantization (notebook 12) is the more deployment-relevant
compression technique** for the STM32 target hardware.

Sparsity levels tested: 20%, 30%, 50%.


In [ ]:
# ── Mount Drive & load utils ────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import sys, shutil, os
UTILS_SRC = "/content/drive/My Drive/thesis/utils"
if os.path.exists(UTILS_SRC):
    shutil.copytree(UTILS_SRC, "/content/utils", dirs_exist_ok=True)
    sys.path.insert(0, "/content")
    print("✅ utils loaded from Drive")
else:
    sys.path.insert(0, "/content")
    print("⚠️  Place the utils/ folder at: My Drive/thesis/utils/")


In [ ]:
# ── Standard imports ────────────────────────────────────────────────
import os, time, random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

from utils.dataset import prepare_dataset, get_loaders, get_test_loader
from utils.models  import (
    VGG_Scratch, VGG_Pretrained,
    ResNet_Scratch, ResNet18_Pretrained,
    MobileNetV2_Scratch, MobileNetV3_Pretrained,
    count_params, model_size_mb, MODEL_REGISTRY,
)
from utils.train import (
    setup_device, set_seed, evaluate,
    train_model, train_model_three_phase,
    train_multi_seed, train_kd, plot_history,
)

device = setup_device(seed=41)


In [ ]:
# ── Dataset setup ───────────────────────────────────────────────────
prepare_dataset()


In [ ]:
SAVE_DIR = "/content/drive/My Drive/Colab Notebooks"


In [ ]:
import torch.nn.utils.prune as prune
import copy

train_loader, val_loader = get_loaders(batch_size=64, augmentation="standard")


In [ ]:
# ── Point to best KD checkpoint ──────────────────────────────────────
PRUNE_CKPT  = f"{SAVE_DIR}/kd_ft_seed_XX.pth"
MODEL_FN    = MobileNetV3_Pretrained

base_model = MODEL_FN().to(device)
base_model.load_state_dict(torch.load(PRUNE_CKPT, map_location=device))
base_acc = evaluate(base_model, val_loader, device)
tp, _ = count_params(base_model)
print(f"Base model accuracy: {base_acc*100:.2f}%  |  Params: {tp/1e6:.2f}M")


In [ ]:
# ── Pruning sweep ─────────────────────────────────────────────────────
from utils.train import train_epoch

FT_EPOCHS    = 5
FT_LR        = 1e-4
PRUNE_LEVELS = [0.20, 0.30, 0.50]

prune_results = []

for amount in PRUNE_LEVELS:
    print(f"\n{'='*50}\nSparsity target: {amount*100:.0f}%")

    model = copy.deepcopy(base_model)
    conv_layers = [(m, "weight") for m in model.modules() if isinstance(m, nn.Conv2d)]

    # Apply L1 unstructured pruning
    for layer, param in conv_layers:
        prune.l1_unstructured(layer, name=param, amount=amount)

    acc_pruned = evaluate(model, val_loader, device)
    print(f"  After pruning  (before FT): {acc_pruned*100:.2f}%")

    # Short fine-tuning
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = torch.optim.Adam(model.parameters(), lr=FT_LR)
    for ep in range(1, FT_EPOCHS + 1):
        tl, ta = train_epoch(model, train_loader, optimizer, criterion, device)
        va = evaluate(model, val_loader, device)
        print(f"  FT {ep}/{FT_EPOCHS}  train={ta*100:.2f}%  val={va*100:.2f}%")

    # Make pruning permanent
    for layer, param in conv_layers:
        prune.remove(layer, param)

    acc_ft = evaluate(model, val_loader, device)

    # Measure actual sparsity
    zeros = sum((p == 0).sum().item() for p in model.parameters())
    total = sum(p.numel()             for p in model.parameters())
    sparsity = zeros / total

    path = f"{SAVE_DIR}/pruned_{int(amount*100)}pct.pth"
    torch.save(model.state_dict(), path)

    prune_results.append({
        "Sparsity target": f"{int(amount*100)}%",
        "Actual sparsity": f"{sparsity*100:.1f}%",
        "Acc after prune (%)": round(acc_pruned*100, 2),
        "Acc after FT (%)":    round(acc_ft*100, 2),
        "Δ vs base (%)":       round((acc_ft - base_acc)*100, 2),
    })
    print(f"  Final: {acc_ft*100:.2f}%  actual sparsity={sparsity*100:.1f}%")


In [ ]:
import pandas as pd
df = pd.DataFrame(prune_results).set_index("Sparsity target")
print("\nPruning Summary")
print(df.to_string())
print(f"\nBase accuracy (no pruning): {base_acc*100:.2f}%")
print("\n⚠️  Reminder: sparsity does not reduce inference time on STM32.")
print("   See notebook 12 for deployment-relevant compression (INT8 quantization).")
